# Superstore product sales: a four-year time series

Companion notebook to the `superstore-sales-timeseries` project. It walks through the same stages the pipeline runs, using the functions in `src/superstore`, so every number here matches `reports/REPORT.md`.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..') / 'src'))
import pandas as pd
import matplotlib.pyplot as plt
from superstore.data import load_raw
from superstore.clean import clean_orders
from superstore.features import weekly_revenue, monthly_revenue, product_summary


## 1. Load and verify the pinned dataset

In [ ]:
raw = load_raw()
orders, audit = clean_orders(raw)
print(audit)
print('rows', len(orders), 'weeks', orders['Week Start'].nunique())


## 2. Weekly revenue series

In [ ]:
weekly = weekly_revenue(orders)
weekly.head()
weekly['revenue'].plot(figsize=(11, 3.5), title='Weekly revenue')
plt.show()


## 3. Decomposition

In [ ]:
from superstore.timeseries import decompose, seasonal_factors, strength
monthly = monthly_revenue(orders)
decomposed = decompose(monthly)
print(seasonal_factors(decomposed).round(3).to_dict())
print(strength(decomposed))


## 4. Product concentration and segments

In [ ]:
from superstore.segments import build_segments, summarise
products = product_summary(orders)
segmented, profile = build_segments(products)
summary = summarise(segmented, profile)
print('ABC', summary['abc_products'], summary['abc_revenue_share'])
print('segments', summary['segments'])


## 5. Weekly anomalies

In [ ]:
from superstore.anomalies import detect_weekly_anomalies, explain_anomalies
scored = detect_weekly_anomalies(weekly)
flagged = scored[scored['is_anomaly']]
explained = explain_anomalies(orders, flagged)
explained[['week_start', 'revenue', 'expected', 'z_score', 'largest_order_value']]


## 6. 13-week forecast

In [ ]:
from superstore.forecast import evaluate_holdout
frame, result = evaluate_holdout(weekly)
payload = result['payload']
print(payload['selected_model'], payload['test'])
frame[['week_start', 'actual', 'predicted', 'lower', 'upper']].round(0)


## 7. Diagnostics

See `reports/figures/19-22` for the model comparison, backtest folds, hold-out forecast and residual diagnostics.